# Selección del modelo

In [1]:
# Cargando las librerias necesarias
import pickle
import pandas as pd
import numpy as np
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, RandomizedSearchCV, HalvingRandomSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, make_scorer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import RFE, SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

# Customizaciones
pd.set_option('display.max_columns', None)

In [2]:
# Cargando los datos de la fase anterior de limpieza desde un fichero pickle
df = pd.read_pickle('./data/df_trans.pkl')

# Verificar el dataframe
print(df.head())
print(df.info())
print(df.shape)

   Diabetes_binary  HighBP  HighChol  CholCheck       BMI  Smoker  Stroke  \
0              0.0     1.0       0.0        1.0 -0.602885     0.0     0.0   
1              0.0     1.0       1.0        1.0 -0.602885     1.0     1.0   
2              0.0     0.0       0.0        1.0 -0.602885     0.0     0.0   
3              0.0     1.0       1.0        1.0 -0.281138     1.0     0.0   
4              0.0     0.0       0.0        1.0 -0.120264     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  HvyAlcoholConsump  \
0                   0.0           1.0     0.0      1.0                0.0   
1                   0.0           0.0     1.0      0.0                0.0   
2                   0.0           1.0     1.0      1.0                0.0   
3                   0.0           1.0     1.0      1.0                0.0   
4                   0.0           1.0     1.0      1.0                0.0   

   AnyHealthcare  NoDocbcCost   GenHlth  MentHlth  PhysHlth  DiffWalk  Sex

## Hacer una selección de variables para reducir el input de los usuarios

In [3]:
X = df.drop(columns=["Diabetes_binary"])
y = df["Diabetes_binary"].astype(int)

# 1. Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X, y)
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

print("=== Importancia de variables (Random Forest) ===")
print(importancias)

# 2. Logistic Regression con L1 (Lasso)
log_l1 = LogisticRegression(penalty="l1", solver="liblinear", random_state=42)
log_l1.fit(X, y)
coefs = pd.Series(np.abs(log_l1.coef_[0]), index=X.columns).sort_values(ascending=False)

print("\n=== Importancia de variables (Logistic L1) ===")
print(coefs)

# 3. Selección de top 10 por RF
top10_rf = importancias.head(10).index.tolist()
# 4. Selección de top 10 por L1
top10_l1 = coefs.head(10).index.tolist()

# Intersección como "más estables"
selecionadas_RF_L1 = list(set(top10_rf) & set(top10_l1))

print("\nVariables recomendadas (intersección RF + L1):", selecionadas_RF_L1)

=== Importancia de variables (Random Forest) ===
BMI                     0.167910
Age                     0.128856
GenHlth                 0.103933
Income                  0.084947
HighBP                  0.072699
PhysHlth                0.069943
Education               0.058834
MentHlth                0.053136
HighChol                0.039540
Smoker                  0.029253
Fruits                  0.029163
Sex                     0.026344
DiffWalk                0.024610
PhysActivity            0.023627
Veggies                 0.022443
HeartDiseaseorAttack    0.019033
NoDocbcCost             0.012313
Stroke                  0.010025
HvyAlcoholConsump       0.009570
AnyHealthcare           0.007652
CholCheck               0.006170
dtype: float64

=== Importancia de variables (Logistic L1) ===
CholCheck               1.312852
HvyAlcoholConsump       0.745138
HighBP                  0.696554
GenHlth                 0.627027
HighChol                0.571138
BMI                     0.5406

In [4]:
target = "Diabetes_binary"
X = df.drop(columns=[target])
y = df[target].astype(int)

# Escalado 0-1 SOLO para la selección (no modifica df)
X_nonneg = MinMaxScaler().fit_transform(X)

selector = SelectKBest(score_func=chi2, k=10)
selector.fit(X_nonneg, y)

selecionadas_KBest = X.columns[selector.get_support()].tolist()
print("Variables seleccionadas (chi²):", selecionadas_KBest)

Variables seleccionadas (chi²): ['HighBP', 'HighChol', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'HvyAlcoholConsump', 'GenHlth', 'PhysHlth', 'DiffWalk', 'Age']


## Hacer el split entre train y test

In [5]:
# Separar variables predictoras (X) y objetivo (y)
# Variables seleccionadas
# vars_seleccionadas = selecionadas_RF_L1
# vars_seleccionadas = selecionadas_KBest
target = 'Diabetes_binary'

# Definir X e y solo con las variables seleccionadas
# X = df[vars_seleccionadas]
X = df.drop(columns=[target])
y = df[target].astype(int)

# Split en train y test (80/20) – sin estratificación (como indicaste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,stratify=y, random_state=99
)

print("Shape train:", X_train.shape, "Shape test:", X_test.shape)
print("\nDistribución target en train:\n", y_train.value_counts(normalize=True).round(3))
print("\nDistribución target en test:\n", y_test.value_counts(normalize=True).round(3))

Shape train: (54174, 21) Shape test: (13544, 21)

Distribución target en train:
 Diabetes_binary
1    0.508
0    0.492
Name: proportion, dtype: float64

Distribución target en test:
 Diabetes_binary
1    0.508
0    0.492
Name: proportion, dtype: float64


## Por Cross validation

In [6]:
# Separar variables predictoras (X) y objetivo (y)
# Variables seleccionadas
# vars_seleccionadas = selecionadas_RF_L1
# vars_seleccionadas = selecionadas_KBest
target = 'Diabetes_binary'

# Definir X e y solo con las variables seleccionadas
# X = df[vars_seleccionadas].copy()
X = df.drop(columns=[target])  # opción: todas las features
y = df[target].astype(int)
print("X shape:", X.shape, "| y shape:", y.shape)

X shape: (67718, 21) | y shape: (67718,)


## Preparar un pipeline para entrenar los modelos candidatos

In [7]:
# Definir pipelines de los modelos

pipelines = {
    "LogReg": Pipeline(steps=[
        ("model", LogisticRegression(max_iter=200, solver="lbfgs"))
    ]),
    "SVM-RBF": Pipeline(steps=[
        ("model", SVC(kernel="rbf", probability=True, random_state=42))  # datos ya escalados
    ]),
    "SVM-Linear": Pipeline(steps=[
        ("model", LinearSVC(random_state=42, max_iter=5000))
    ]),
    "RF": Pipeline(steps=[
        ("model", RandomForestClassifier(n_estimators=300, random_state=42))
    ]),
    "GB": Pipeline(steps=[
        ("model", GradientBoostingClassifier(random_state=42))
    ]),
      "HistGB": Pipeline(steps=[
        ("model", HistGradientBoostingClassifier(random_state=42))
    ]),
    "ExtraTrees": Pipeline(steps=[
        ("model", ExtraTreesClassifier(n_estimators=400, random_state=42))
    ]),
    # Lineales adicionales
    "SVM-Linear-Cal": Pipeline(steps=[
        ("model", CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=5000), cv=5))
    ]),
    "SGD-Log": Pipeline(steps=[
        ("model", SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=2000, random_state=42))
    ]),
    "LogReg-Bal": Pipeline(steps=[
        ("model", LogisticRegression(max_iter=500, solver="lbfgs", class_weight="balanced"))
    ]),
}

## Crear la función para evaluar los modelos

In [8]:
# Función para evaluar modelos por split train/test

def evaluar_modelo_split(nombre, pipe, Xtr, ytr, Xte, yte, average="binary"):
    """
    Entrena y evalúa un pipeline en un split train/test.
    Devuelve un dict con métricas para poder rankear modelos.
    Criterio de ranking: AvgPrecision (PR-AUC) -> F1.
    """
    pipe.fit(Xtr, ytr)
    y_pred = pipe.predict(Xte)

    # Scores continuos para AUC/PR-AUC
    model_step = pipe.named_steps["model"]
    if hasattr(model_step, "predict_proba"):
        y_score = pipe.predict_proba(Xte)[:, 1]
    elif hasattr(model_step, "decision_function"):
        y_score = pipe.decision_function(Xte)
    else:
        y_score = None  # no hay score continuo

    # Métricas
    acc  = accuracy_score(yte, y_pred)
    prec = precision_score(yte, y_pred, average=average, zero_division=0)
    rec  = recall_score(yte, y_pred, average=average, zero_division=0)
    f1   = f1_score(yte, y_pred, average=average, zero_division=0)
    auc  = roc_auc_score(yte, y_score) if y_score is not None else np.nan
    ap   = average_precision_score(yte, y_score) if y_score is not None else np.nan  # PR-AUC

    print(f"\n=== {nombre} ===")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | ROC-AUC: {auc:.4f} | PR-AUC: {ap:.4f}")
    print(classification_report(yte, y_pred, digits=4))

    return {
        "Modelo": nombre,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC-AUC": auc,
        "AvgPrecision(PR-AUC)": ap
    }

In [9]:
# Función para evaluar modelos por cross-validation

def evaluar_modelo_cv(nombre, pipe, X, y, cv_splits=5, stratified=False, average="binary"):
    """
    Imprime classification_report por fold y devuelve métricas promedio.
    Selección posterior se hará por Average Precision (PR-AUC) y F1.
    """
    if stratified:
        from sklearn.model_selection import StratifiedKFold
        kf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
        splitter = kf.split(X, y)
    else:
        kf = KFold(n_splits=cv_splits, shuffle=True, random_state=42)
        splitter = kf.split(X)

    registros = []
    fold = 1
    for idx_tr, idx_te in splitter:
        X_tr, X_te = X.iloc[idx_tr], X.iloc[idx_te]
        y_tr, y_te = y.iloc[idx_tr], y.iloc[idx_te]

        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        # Scores continuos para AUC y Average Precision
        if hasattr(pipe.named_steps["model"], "predict_proba"):
            y_score = pipe.predict_proba(X_te)[:, 1]
        elif hasattr(pipe.named_steps["model"], "decision_function"):
            y_score = pipe.decision_function(X_te)
        else:
            y_score = None

        print(f"\n===== {nombre} | Fold {fold}/{cv_splits} =====")
        print(classification_report(y_te, y_pred, digits=4))

        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, average=average, zero_division=0)
        rec  = recall_score(y_te, y_pred, average=average, zero_division=0)
        f1   = f1_score(y_te, y_pred, average=average, zero_division=0)
        auc  = roc_auc_score(y_te, y_score) if y_score is not None else np.nan
        ap   = average_precision_score(y_te, y_score) if y_score is not None else np.nan  # PR-AUC

        registros.append({
            "fold": fold, "accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "roc_auc": auc, "avg_precision": ap
        })
        fold += 1

    df_cv = pd.DataFrame(registros)
    resumen = df_cv.mean(numeric_only=True).to_dict()
    print("\n===== Resumen CV (promedios) =====")
    print(pd.Series(resumen).round(4))
    return resumen, df_cv

## Entrenar y evaluar los modelos

In [10]:
# Entrenando y evaluando los modelos por split train/test

resultados_split = []

# Entrenando y evaluando los modelos por split train/test
for nombre, pipe in pipelines.items():
    m = evaluar_modelo_split(nombre, pipe, X_train, y_train, X_test, y_test, average="binary")
    resultados_split.append(m)

# Ranking según el mismo criterio que en CV: PR-AUC primero, F1 después
df_rank_split = pd.DataFrame(resultados_split).sort_values(
    by=["AvgPrecision(PR-AUC)", "F1"],
    ascending=False
).reset_index(drop=True)

print("\n===== Ranking (split train/test) — Prioridad: PR-AUC → F1 =====")
print(df_rank_split.round(4).to_string(index=False))

mejor_split = df_rank_split.iloc[0]["Modelo"]
print(f"\n🏆 Mejor modelo en split (criterio PR-AUC → F1): {mejor_split}")


=== LogReg ===
Accuracy: 0.7400 | Precision: 0.7343 | Recall: 0.7652 | F1: 0.7494 | ROC-AUC: 0.8174 | PR-AUC: 0.7976
              precision    recall  f1-score   support

           0     0.7464    0.7140    0.7299      6662
           1     0.7343    0.7652    0.7494      6882

    accuracy                         0.7400     13544
   macro avg     0.7404    0.7396    0.7397     13544
weighted avg     0.7403    0.7400    0.7398     13544


=== SVM-RBF ===
Accuracy: 0.7425 | Precision: 0.7208 | Recall: 0.8051 | F1: 0.7607 | ROC-AUC: 0.8089 | PR-AUC: 0.7680
              precision    recall  f1-score   support

           0     0.7710    0.6779    0.7215      6662
           1     0.7208    0.8051    0.7607      6882

    accuracy                         0.7425     13544
   macro avg     0.7459    0.7415    0.7411     13544
weighted avg     0.7455    0.7425    0.7414     13544


=== SVM-Linear ===
Accuracy: 0.7399 | Precision: 0.7315 | Recall: 0.7711 | F1: 0.7508 | ROC-AUC: 0.8173 | PR

In [11]:
# Entranamiento y evaluación por CV

resultados = []

# Entrenando y evaluando los modelos por CV
for nombre, pipe in pipelines.items():
    resumen, _df_folds = evaluar_modelo_cv(
        nombre, pipe, X, y,
        cv_splits=5, stratified=True, average="binary"  # pon stratified=True si lo prefieres
    )
    # Guardamos métricas clave para ranking
    resultados.append({
        "Modelo": nombre,
        "AvgPrecision(PR-AUC)": resumen.get("avg_precision", np.nan),
        "F1": resumen.get("f1", np.nan),
        "Precision": resumen.get("precision", np.nan),
        "Recall": resumen.get("recall", np.nan),
        "ROC-AUC": resumen.get("roc_auc", np.nan),
        "Accuracy": resumen.get("accuracy", np.nan),
    })

# Ranking de modelos (criterio: 1) AvgPrecision(PR-AUC), 2) F1)
df_rank = pd.DataFrame(resultados)
df_rank = df_rank.sort_values(
    by=["AvgPrecision(PR-AUC)", "F1"],
    ascending=False
).reset_index(drop=True)

print("\n===== Ranking de modelos (prioridad: PR-AUC, luego F1) =====")
print(df_rank.round(4).to_string(index=False))

mejor_modelo = df_rank.iloc[0]["Modelo"]
print(f"\n🏆 Mejor modelo por criterio PR-AUC → F1: {mejor_modelo}")


===== LogReg | Fold 1/5 =====
              precision    recall  f1-score   support

           0     0.7560    0.7101    0.7324      6662
           1     0.7350    0.7781    0.7559      6882

    accuracy                         0.7447     13544
   macro avg     0.7455    0.7441    0.7441     13544
weighted avg     0.7453    0.7447    0.7443     13544


===== LogReg | Fold 2/5 =====
              precision    recall  f1-score   support

           0     0.7523    0.7225    0.7371      6662
           1     0.7413    0.7697    0.7552      6882

    accuracy                         0.7465     13544
   macro avg     0.7468    0.7461    0.7461     13544
weighted avg     0.7467    0.7465    0.7463     13544


===== LogReg | Fold 3/5 =====
              precision    recall  f1-score   support

           0     0.7521    0.7154    0.7333      6662
           1     0.7369    0.7717    0.7539      6882

    accuracy                         0.7440     13544
   macro avg     0.7445    0.7436  

## Fine tunning del modelo ganador

### Fine tunning 1 - Grid Search

In [12]:
# Modelo base
modelo_ganador = GradientBoostingClassifier(random_state=42)

param_grid = {
    "n_estimators":      [200, 400, 600],
    "learning_rate":     [0.01, 0.05, 0.1],
    "max_depth":         [2, 3, 4],
    "subsample":         [0.7, 0.9, 1.0],
    "max_features":      ["sqrt", "log2", None],
    "min_samples_leaf":  [1, 5, 10],
    "min_samples_split": [2, 5, 10],
    # Para evitar un grid enorme, desactivo early stopping en la búsqueda.
    # Si quieres probarlo, añade: "n_iter_no_change": [5, 10] y "validation_fraction":[0.1, 0.2]
    "n_iter_no_change":  [None],
}

# Métricas
scorers = {
    "ap": "average_precision",  # PR-AUC
    "f1": "f1"
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=modelo_ganador,
    param_grid=param_grid,
    scoring=scorers,
    refit="ap",                # selecciona por PR-AUC (criterio principal)
    cv=cv,
    verbose=1,
    n_jobs=-1,
    error_score="raise"
)

grid_search.fit(X_train, y_train)

print("Mejores hiperparámetros:", grid_search.best_params_)
print("Mejor PR-AUC CV:", grid_search.best_score_)

# Evaluación en test
best_gb = grid_search.best_estimator_

y_pred  = best_gb.predict(X_test)
y_score = best_gb.predict_proba(X_test)[:, 1]

print("\n=== Test ===")
print("PR-AUC   :", average_precision_score(y_test, y_score))
print("F1       :", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))
print("Accuracy :", accuracy_score(y_test, y_pred))

Fitting 5 folds for each of 2187 candidates, totalling 10935 fits


KeyboardInterrupt: 

### Fine tunning 2 - Random Grid Search

In [ ]:
# Fine Tunning del modelo ganador (Gradient Boosting)

# Búsqueda aleatoria (GridSearch)
# Modelo base
modelo_ganador = GradientBoostingClassifier(random_state=42)

# Espacio de búsqueda
param_dist = {
    "n_estimators":      np.arange(100, 1201, 100),
    "learning_rate":     np.logspace(-3, -0.3, 10),   # ~0.001–0.5 aprox
    "max_depth":         [2, 3, 4, 5],
    "subsample":         [0.6, 0.7, 0.8, 0.9, 1.0],
    "max_features":      ["sqrt", "log2", None, 0.5, 0.8],
    "min_samples_leaf":  [1, 2, 5, 10, 20],
    "min_samples_split": [2, 5, 10, 20],
    "n_iter_no_change":  [None, 5, 10],
    "validation_fraction": [0.1, 0.15, 0.2],
}

# Metricas
scorers = {
    "ap": "average_precision",  # PR-AUC
    "f1": "f1"
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rand_search = RandomizedSearchCV(
    estimator=modelo_ganador,
    param_distributions=param_dist,
    n_iter=60,                 
    scoring=scorers,
    refit="ap",                # selecciona por PR-AUC (criterio principal)
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42,
    error_score="raise"        # útil para ver errores reales si aparecen
)

rand_search.fit(X_train, y_train)

print("Mejores hiperparámetros:", rand_search.best_params_)
print("Mejor PR-AUC CV:", rand_search.best_score_)

# Evaluación en test
best_gb = rand_search.best_estimator_


y_pred  = best_gb.predict(X_test)
y_score = best_gb.predict_proba(X_test)[:, 1]

print("\n=== Test ===")
print("PR-AUC   :", average_precision_score(y_test, y_score))
print("F1       :", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))
print("Accuracy :", accuracy_score(y_test, y_pred))


### Fine tunning 3 - Halving Random Grid Search

In [ ]:
gb = GradientBoostingClassifier(random_state=42)

# ¡OJO! No incluir 'n_estimators' aquí porque es el 'resource'
param_dist_small = {
    "learning_rate":    np.logspace(-3, -0.3, 8),
    "max_depth":        [2, 3, 4, 5],
    "subsample":        [0.6, 0.8, 1.0],
    "max_features":     ["sqrt", None],
    "min_samples_leaf": [1, 5, 10],
    "n_iter_no_change": [None, 5],
    "validation_fraction": [0.1, 0.2],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

halving = HalvingRandomSearchCV(
    estimator=gb,
    param_distributions=param_dist_small,
    factor=3,
    resource="n_estimators",   # el recurso que irá aumentando
    min_resources=100,         # punto de partida
    max_resources=1200,        # máximo de árboles
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

halving.fit(X_train, y_train)

print("Mejores hiperparámetros (PR-AUC CV):", halving.best_params_)
print("Mejor PR-AUC CV:", halving.best_score_)

# ===== Evaluación en TEST =====
best_gb = halving.best_estimator_
y_pred  = best_gb.predict(X_test)
y_score = best_gb.predict_proba(X_test)[:, 1]

print("\n=== Test ===")
print("PR-AUC   :", average_precision_score(y_test, y_score))
print("F1       :", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))
print("Accuracy :", accuracy_score(y_test, y_pred))

### Ajustando aun mas

In [ ]:
# === 1) Punto de partida (trae esto de tu búsqueda previa) ===
# Ejemplos:
best_params_base = rand_search.best_params_
# best_params_base = halving.best_params_
'''
best_params_base = {
    "learning_rate": 0.05,
    "max_depth": 3,
    "subsample": 0.8,
    "max_features": "sqrt",
    "min_samples_leaf": 5,
    "min_samples_split": 10,
    "n_iter_no_change": 5,
    "validation_fraction": 0.1,
    "n_estimators": 600,            
}
'''
# === 2) Definir un grid “fino” alrededor de esos valores ===
def around(val, factors=(0.5, 1.0, 1.5), clip_min=None, clip_max=None):
    arr = sorted(set([val*f for f in factors]))
    if clip_min is not None: arr = [max(clip_min, x) for x in arr]
    if clip_max is not None: arr = [min(clip_max, x) for x in arr]
    return arr

grid = {
    "learning_rate": around(best_params_base["learning_rate"], (0.5, 1.0, 1.5), 1e-3, 0.5),
    "max_depth":     sorted(set([max(2, best_params_base["max_depth"]-1),
                                 best_params_base["max_depth"],
                                 best_params_base["max_depth"]+1])),
    "subsample":     [max(0.6, best_params_base["subsample"]-0.1),
                      best_params_base["subsample"],
                      min(1.0, best_params_base["subsample"]+0.1)],
    "max_features":  [best_params_base["max_features"]],  # o prueba ["sqrt", "log2", None]
    "min_samples_leaf": [max(1, best_params_base["min_samples_leaf"]//2),
                         best_params_base["min_samples_leaf"],
                         best_params_base["min_samples_leaf"]*2],
    "min_samples_split": [max(2, best_params_base["min_samples_split"]//2),
                          best_params_base["min_samples_split"],
                          min(100, best_params_base["min_samples_split"]*2)],
    # Mantén early stopping si te funcionó bien:
    "n_iter_no_change": [best_params_base.get("n_iter_no_change", None)],
    "validation_fraction": [best_params_base.get("validation_fraction", 0.1)],
    # Incluye n_estimators si NO estás usando Halving como recurso:
    "n_estimators": [max(100, int(best_params_base.get("n_estimators", 600)*f)) for f in (0.75, 1.0, 1.25)],
}

# === 3) Instanciar modelo base con los mejores parámetros previos ===
gb_base = GradientBoostingClassifier(random_state=42, **best_params_base)

# === 4) GridSearchCV: refit por PR-AUC (Average Precision) ===
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gridcv = GridSearchCV(
    estimator=gb_base,
    param_grid=grid,
    scoring={"ap": "average_precision", "f1": "f1"},
    refit="ap",               # selecciona por PR-AUC (tu criterio principal)
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
gridcv.fit(X_train, y_train)

print("🔧 Mejor combinación (CV, PR-AUC):", gridcv.best_params_)
print("🏅 Mejor PR-AUC CV:", gridcv.best_score_)

# === 5) Evaluación en test del mejor modelo ===
best_gb = gridcv.best_estimator_
y_pred  = best_gb.predict(X_test)
y_score = best_gb.predict_proba(X_test)[:, 1]

print("\n=== Métricas en TEST ===")
print("PR-AUC   :", average_precision_score(y_test, y_score))
print("F1       :", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))
print("Accuracy :", accuracy_score(y_test, y_pred))